In [1]:
import sys
sys.path.append("..")

from getpass import getpass
from pathlib import Path

import pandas as pd
from sqlalchemy import URL, create_engine

from src.features import add_features
from src.split import make_masks

In [2]:
password = getpass("Password: ")

db_url = URL.create(
    drivername="oracle+oracledb",
    username="fraud_user",
    password=password,
    host="localhost",
    port=1522,
    query={"service_name": "FREEPDB1"},
)
engine = create_engine(db_url)

In [3]:
query = """
SELECT
    step,
    transaction_type,
    amount,
    old_balance_orig,
    new_balance_orig,
    old_balance_dest,
    new_balance_dest,
    is_fraud,
    name_orig,
    name_dest,
    is_flagged_fraud
FROM transactions_risk
"""

data_chunks = []

for chunk in pd.read_sql_query(query, con=engine, chunksize=200_000):
    chunk.columns = chunk.columns.str.lower()

    chunk = chunk.astype({
        "step": "int16",
        "transaction_type": "category",
        "amount": "float64",
        "old_balance_orig": "float64",
        "new_balance_orig": "float64",
        "old_balance_dest": "float64",
        "new_balance_dest": "float64",
        "is_fraud": "int8",
        "name_orig": "string",
        "name_dest": "string",
        "is_flagged_fraud": "int8",
    })

    data_chunks.append(chunk)

risk_df = pd.concat(data_chunks, ignore_index=True)
del data_chunks

In [4]:
print(risk_df.shape)
print(risk_df.isna().sum())
print(risk_df["is_fraud"].value_counts())
risk_df.head()

(2770409, 11)
step                0
transaction_type    0
amount              0
old_balance_orig    0
new_balance_orig    0
old_balance_dest    0
new_balance_dest    0
is_fraud            0
name_orig           0
name_dest           0
is_flagged_fraud    0
dtype: int64
is_fraud
0    2762196
1       8213
Name: count, dtype: int64


,step,transaction_type,amount,old_balance_orig,new_balance_orig,old_balance_dest,new_balance_dest,is_fraud,name_orig,name_dest,is_flagged_fraud
0,1,TRANSFER,181.00,181.0,0.0,0.0,0.00,1,C1305486145,C553264065,0
1,1,CASH_OUT,181.00,181.0,0.0,21182.0,0.00,1,C840083671,C38997010,0
2,1,CASH_OUT,229133.94,15325.0,0.0,5083.0,51513.44,0,C905080434,C476402209,0
3,1,TRANSFER,215310.30,705.0,0.0,22425.0,0.00,0,C1670993182,C1100439041,0
4,1,TRANSFER,311685.89,10835.0,0.0,6267.0,2719172.89,0,C1984094095,C932583850,0


In [5]:
model_df = add_features(risk_df)

model_df = model_df.drop(
    columns=["new_balance_orig", "new_balance_dest", "is_flagged_fraud"]
)

assert len(model_df) == len(risk_df)
model_df.dtypes

step                              int16
transaction_type               category
amount                          float64
old_balance_orig                float64
old_balance_dest                float64
is_fraud                           int8
name_orig                        string
name_dest                        string
hour                               int8
hour_sin                        float32
hour_cos                        float32
log_amount                      float32
log_old_balance_orig            float32
log_old_balance_dest            float32
origin_zero_balance                int8
destination_zero_balance           int8
log_amount_to_balance_ratio     float32
full_balance_attempt               int8
dtype: object

In [6]:
train_mask, val_mask, test_mask = make_masks(model_df)

for name, mask in [("Train", train_mask), ("Validation", val_mask), ("Test", test_mask)]:
    part = model_df.loc[mask]
    print(
        f"{name:10s} rows: {len(part):>9,}  "
        f"steps: {part['step'].min()}-{part['step'].max()}  "
        f"fraud: {int(part['is_fraud'].sum()):>5,}  "
        f"fraud share: {part['is_fraud'].mean():.4%}"
    )

Train      rows: 1,951,895  steps: 1-323  fraud: 3,643  fraud share: 0.1866%
Validation rows:   410,437  steps: 324-377  fraud:   560  fraud share: 0.1364%
Test       rows:   408,077  steps: 378-743  fraud: 4,010  fraud share: 0.9827%


In [7]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
model_df.to_parquet("../data/processed/model_df.parquet")
print("Saved:", model_df.shape)

Saved: (2770409, 18)
